# TIME SERIES SPLIT
Basically, adjust parameters, run the cell with the Regressor in it, then run the testing loop. Get the results once done. Go back, readjust parameters, run it, then back and so forth.

In [ ]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive')

!pip install xgboost --quiet
# Import libraries/modules to use
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, median_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
import joblib as jb
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

import torch
import random
import os

def set_seed(seed=42):
    """Sets the seed for reproducibility."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# Call the function
set_seed(42)

# Load in the data
df_pivoted = pd.read_csv('/content/drive/MyDrive/ACP_Project_28A/PivotedData.csv')

#weekends
df_pivoted['is_weekend'] = df_pivoted['day_of_week'].isin([6, 7]).astype(int)

# Convert date to datetime before trying to resort
df_pivoted['date'] = pd.to_datetime(df_pivoted['date'])

# Enforce that the pivoted dataframe must be ordered chronologically and drop the residual old index
df_pivoted = df_pivoted.sort_values(['date', 'hour']).reset_index(drop=True)

# Convert station_key to categorical before encoding
df_pivoted['station_key'] = df_pivoted['station_key'].astype(str)

#display(df_pivoted)
# Extract the feature set (X) and the target (y)
X = df_pivoted[['station_key', 'traffic_direction_seq', 'cardinal_direction_seq', 'month', 'day_of_week', 'school_holiday', 'hour', 'avg_prev3h', 'avg_prevday','is_weekend']]
y = df_pivoted['traffic_vol']

#cyclical encoding for hours
X['hour_sin'] = np.sin(2 * np.pi * X['hour'] / 24)
X['hour_cos'] = np.cos(2 * np.pi * X['hour'] / 24)

# One-hot encoder to encode the station_key into a numerical matrix at training, testing and actual use time - feed into Pipeline
preprocessor = ColumnTransformer(transformers=[('station_encode', OneHotEncoder(handle_unknown='ignore'), ['station_key'])], remainder='passthrough')

XGBRegressor = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.5,
        reg_alpha=0.5,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1
    ))
])
# Create TimeSeriesSplit for cross-validation
tss = TimeSeriesSplit(n_splits=5)

# Keep track of the mean absolute error with each iteration
mae_track = []
medae_track = []
rmse_track = []


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_2019/928821144.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['hour_sin'] = np.sin(2 * np.pi * X['hour'] / 24)


In [ ]:
%%time
# For each split from the TimeSeriesSplit, extract train and test sets for features and target, train an interim XGBRegressor, check its predicted values against the actual values and append the mean absolute error between the actual and predicted values to the track
for train_sub, test_sub in tss.split(X):
    X_train, X_test = X.iloc[train_sub], X.iloc[test_sub]
    y_train, y_test = y.iloc[train_sub], y.iloc[test_sub]
    # Log-transform training targets
    y_train_log = np.log1p(y_train)
    print('Log transform applied to targets!')
    # Train a test/sample model with current parameters
    XGBRegressor.fit(X_train, y_train_log)
    print('Test model trained!')
    # Get predictions from the now trained model to test
    predictions_log = XGBRegressor.predict(X_test)
    predictions = np.expm1(predictions_log)
    print('Exp transform applied to predictions!')
    # Tack on the mean absolute error onto the track
    mae_track.append(mean_absolute_error(y_test, predictions))
    medae_track.append(median_absolute_error(y_test, predictions))
    rmse_track.append(root_mean_squared_error(y_test, predictions))
    print('Scores added to tracks!')

# Show the running track for evaluation
metrics = pd.DataFrame(columns=['0', '1', '2', '3', '4'], index=['mae', 'medae', 'rmse'])
metrics.loc['mae'] = mae_track
metrics.loc['medae'] = medae_track
metrics.loc['rmse'] = rmse_track
avg_metrics = metrics.mean(axis=1)
metrics['Avg'] = avg_metrics

display(metrics)
print("Mean traffic volume: ", df_pivoted['traffic_vol'].mean())
print("Stdev traffic volume: ", df_pivoted['traffic_vol'].std())
print("Min traffic volume: ", df_pivoted['traffic_vol'].min())
print("Q1 traffic volume: ", df_pivoted['traffic_vol'].quantile(0.25))
print("Q2 traffic volume: ", df_pivoted['traffic_vol'].quantile(0.5))
print("Q3 traffic volume: ", df_pivoted['traffic_vol'].quantile(0.75))
print("Max traffic volume: ", df_pivoted['traffic_vol'].max())

In [ ]:
# With parameters figured out, train the final production model - log-transform before training; remember to exp-transform predictions
def train_model(model, X, y):
  y_log = np.log1p(y)
  model.fit(X, y_log)
  return model

In [ ]:
export = train_model(XGBRegressor, X, y)

# Package and export the trained model
jb.dump(export, '/content/drive/MyDrive/ACP_Project_28A/XGBRegressor.joblib', compress=('gzip', 3))

['/content/drive/MyDrive/ACP_Project_28A/XGBRegressor.joblib']

## RESULTS

First run of TimeSeries Split
- MAE: 139.39
- MedAE: 31.64
- RMSE: 304.53

Changed max_depth from 10 to 5
- MAE: 184.64
- MedAE: 54.39
- RMSE: 362.89

Changed n_estimators from 1000 to 500, changed max_depth back to 10
- MAE: 147.61
- MedAE: 36.56
- RMSE: 309.40

Changed reg_lambda from 1.5 to 3 and reg_alpha from 0.5 to 1
- MAE: 147.82
- MedAE: 36.71
- RMSE: 309.34

Added is_weekend column
- MAE: 138.29
- MedAE: 31.44
- RMSE: 300.67

Added cyclical encoding for hours
- MAE: 135.17
- MedAE: 30.41
- RMSE: 297.14

Added cyclical encoding for months
- MAE: 139.49
- MedAE: 30.76
- RMSE: 309.83

Added weekend_hour column and removed cyclical encoding for months
- MAE: 136.03
- MedAE: 30.76
- RMSE: 303.92

I think the best model so far is the model with cyclical encoding for hours

# STANDARD TRAINING METHOD

Goals for Model:
- a generalised predictor that provides estimates for road traffic demand at discrete locations in the Sydney metropolitan area per hour
- the dataset includes the individual collection/monitoring stations and their associated locations, allowing the applicability of the model for each location.


check if gpu is still around

In [ ]:
import torch
torch.cuda.is_available()

## MOUNT DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## IMPORT PACKAGES

In [ ]:
!pip install xgboost --quiet

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

making sure that each run of the colab is consistent

In [ ]:
import random
import os

def set_seed(seed=42):
    """Sets the seed for reproducibility."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# Call the function
set_seed(42)

## DATA

### Load Dataset

In [ ]:
# Load the new pivoted dataset
df_pivoted = pd.read_csv('/content/drive/MyDrive/ACP Project 28A/PivotedData.csv')

# convert date to datetime format
df_pivoted['date'] = pd.to_datetime(df_pivoted['date'])

# force chronological reordering
df_pivoted = df_pivoted.sort_values(['date', 'hour']).reset_index(drop=True)

# convert station_key to string for categorical treatment
df_pivoted['station_key'] = df_pivoted['station_key'].astype(str)

display(df_pivoted.head())

### Testing

- Start off with columns from XGBRegressor.ipynb
- maybe if it is a weekend, weekends can impact traffic flows
- traffic exactly a week ago
- maybe log it to help the skewness, outliers (in splitting)
- cyclical encoding to help model understand time (in splitting)



In [ ]:
#weekends
df_pivoted['is_weekend'] = df_pivoted['day_of_week'].isin([6, 7]).astype(int)

#traffic a week ago
"""
df_pivoted['date_lag_target'] = df_pivoted['date'] - pd.Timedelta(days=7)

df_ref = df_pivoted[['date', 'station_key', 'hour', 'traffic_vol']].copy()
df_ref.columns = ['date', 'station_key', 'hour', 'lag_1week']

df_pivoted = df_pivoted.merge(
    df_ref,
    left_on=['date_lag_target', 'station_key', 'hour'],
    right_on=['date', 'station_key', 'hour'],
    how='left',
    suffixes=('', '_ref')
)

df_pivoted = df_pivoted.drop(columns=['date_lag_target', 'date_ref'])
#df_pivoted = df_pivoted.drop(columns=['is_weekend', 'lag_1week'])
#display(df_pivoted.head())"""

display(df_pivoted.head())

### Splitting

In [ ]:
 #target
  #y = df_pivoted['traffic_vol']
 #log to help skewness
y = np.log1p(df_pivoted['traffic_vol'])

# define features
feature_cols = ['station_key', 'hour', 'month', 'day_of_week','avg_prev3h', 'avg_prevday','is_weekend']
X = df_pivoted[feature_cols]

#cyclical encoding
#X['hour_sin'] = np.sin(2 * np.pi * X['hour'] / 24)
#X['hour_cos'] = np.cos(2 * np.pi * X['hour'] / 24)

# split 70/15/15
n = len(X)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['station_key'])
    ],
    remainder='passthrough'
)


## BUILD XGBRegressor


In [ ]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.5,
        reg_alpha=0.5,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1,
        early_stopping_rounds=50
    ))
])

## TRAINING


In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_val_transformed = preprocessor.transform(X_val)

regressor = model_pipeline.named_steps['regressor']
regressor.fit(
    X_train_transformed,
    y_train,
    eval_set=[(X_val_transformed, y_val)],
    verbose=False
)

### Testing


see what the prediction model is seeing, see if it is correct, visualise it

In [ ]:
y_pred_log = model_pipeline.predict(X_test)

y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

#X_test_transformed = preprocessor.transform(X_test)
#y_pred = regressor.predict(X_test_transformed)

results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

display(results.sample(10))

## EVALUATION METRICS

- Mean Absolute Error (MAE): on average, how much am I off by
- Root Mean Squared Error (RMSE): penalises large errors more so more important in high traffic areas
- R^2 score: how well does the model explain variation in traffic
- MedAE (Median Absolute Error)

In [ ]:
mae = mean_absolute_error(y_test_actual, y_pred)
rmse = mean_squared_error(y_test_actual, y_pred) ** 0.5
r2 = r2_score(y_test_actual, y_pred)
med_ae = median_absolute_error(y_test_actual, y_pred)

#mae = mean_absolute_error(y_test, y_pred)
#rmse = mean_squared_error(y_test, y_pred) ** 0.5
#r2 = r2_score(y_test, y_pred)
#med_ae = median_absolute_error(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2 Score: {r2:.2f}")
print(f"MedAE: {med_ae:.2f}")

## RESULTS

With PivotedData on its own (both avg_prev3h and avg_prevday)
- MAE: 167.75
- RMSE: 326.38
- R2 Score: 0.61

Added historical_hour_day_baseline column
- MAE: 162.75 (improved)
- RMSE: 328.24 (worsened)
- R2 Score: 0.60 (worsened)

Removed avg_prev3h and avg_prevday columns
- MAE: 312.78 (worsened)
- RMSE: 457.54 (worsened)
- R2 Score: 0.22 (worsened)

Removed avg_prevday column from PivotedData
- MAE: 196.08 (improved)
- RMSE: 361.53 (improved)
- R2 Score: 0.52 (improved)

Removed avg_prev3h column from PivotedData, compared from removed avg_prev3h and avg_prevday columns
- MAE: 291.18 (improved)
- RMSE: 428.32 (improved)
- R2 Score: 0.32 (improved)

Adding weekend column compared with PivotedData on its own
- MAE: 152.85 (improved)
- RMSE: 300.61 (improved)
- R2 Score: 0.65 (improved)

Adding log
- MAE: 188.99
- RMSE: 401.54
- R2 Score: 0.40
- MedAE: 42.41

## SAVE AND LOAD MODEL

Job lib file when finished model to package it up
- Job lib dump -> to export file
- Name of variable of model and name it

Load it in later then job.load
- Load in the file that you dumped
